In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet("../data/processed/stratified_sample_full")

In [ ]:
df

In [ ]:
df_topics = pd.read_parquet("../reports/topic_modeling/best_result_reduced_p/all-distilroberta-v1/post_topics.parquet")

In [ ]:
df_topics

In [ ]:
df_topics.groupby('id').size()

In [ ]:
# 1. Prova real: verifica se o tamanho de ambos é igual e se TODOS os IDs batem linha por linha
if len(df) == len(df_topics) and (df['id'] == df_topics['id']).all():
    print("A ordem e os IDs são 100% idênticos!")
    
    # 2. Como a ordem é idêntica, basta copiar a coluna de um para o outro
    df['topic'] = df_topics['topic']
    print("Coluna 'topic' adicionada com sucesso ao DataFrame original!")
    
else:
    print("Cuidado! Alguma linha está fora de ordem ou os tamanhos são diferentes.")

In [ ]:
df_topics['group_name'] = df['group_name']

In [ ]:
import json

caminho_json = "../reports/topic_modeling/macro_topics.json"
with open(caminho_json, 'r', encoding='utf-8') as f:
    json_data = json.load(f)
mapa_original = json_data.get("topic_id_to_macro", {})
mapa_inteiros = {int(k): v for k, v in mapa_original.items()}

if -1 not in mapa_inteiros:
    mapa_inteiros[-1] = "OUTLIERS"

df_topics['macro_topic'] = df_topics['topic'].map(mapa_inteiros)

In [ ]:
df_topics

In [ ]:
resumo_macro = df_topics.groupby('macro_topic').agg(
    total_mensagens=('id', 'size'),       
    total_topicos=('topic', 'nunique')    
).sort_values(by='total_mensagens', ascending=False) # Ordena do maior para o menor

display(resumo_macro)

In [ ]:
macro_topics_info = json_data.get("macro_topics", {})
mapa_politica = {}
for macro_nome, info in macro_topics_info.items():
    # Extrai o valor booleano, assumindo False por padrão caso não exista
    is_political = info.get("Politics_related", False)
    
    if is_political:
        mapa_politica[macro_nome] = "Politic"
    else:
        mapa_politica[macro_nome] = "Non Politic"
mapa_politica["OUTLIERS"] = "Outliers"

In [ ]:
df_topics['political_category'] = df_topics['macro_topic'].map(mapa_politica)

In [ ]:
print("=== Distribuição das Categorias Políticas ===")
distribuicao = df_topics['political_category'].value_counts()
print(distribuicao)

In [ ]:
df_topics

In [ ]:
df_topics.to_parquet("../data/processed/topic_modeling_result.parquet")

In [ ]:
df = pd.read_parquet("../data/processed/topic_modeling_result.parquet")

In [ ]:
df

In [ ]:
# Supondo que o seu DataFrame se chame df_merged
distribuicao_percentual = df['political_category'].value_counts(normalize=True) * 100

# Exibindo o resultado formatado com duas casas decimais
print(distribuicao_percentual.round(2).astype(str) + '%')